# Z5 all-pinned comparison — ideal pin-jointed lattice

This notebook repeats the same three Z5 load cases used in `05_z5_three_case_comparison.ipynb`, but deliberately changes the connection idealization:

> **Every structural member is treated as a pin-ended, axial-only member.**

The original CSV files are **not modified**. The notebook reads the normal Z5 model, makes an analysis-only copy, preserves the original member type in `original_element_type`, and then sets every member's analysis `element_type` to `"truss"`.

The purpose is to isolate the effect of the rigid-joint/frame assumption used in the mixed frame/truss model. The same three load cases are solved through:

1. the project's in-house solver, where every member becomes a true 3D truss/bar element;
2. the PyNite adapter, where every member is represented using the adapter's truss-like pin-ended member formulation.

The main comparison question remains:

> **Which elements are more highly loaded in Case 1 than in both Case 2 and Case 3?**

Because this is an all-truss model, the physically relevant member demand is primarily **axial tension/compression**. Bending demand should be zero (or numerical noise only) in the in-house solution and essentially zero in the PyNite truss-like solution.

For direct solver-to-solver comparison this notebook keeps:

```python
SECTION_MODE = "isotropic"
CROSSING_DIAGONAL_MODE = "continuous"
```

The section inertia choice no longer affects the true in-house truss stiffness, but it is retained so the PyNite translation and result tables remain directly comparable with the previous notebook.

## Modelling assumptions and what “all joints pinned” means here

This notebook intentionally makes a stronger idealization than the mixed frame/truss model.

- **All 104 input members are reclassified as truss members in an analysis-only copy.** After the existing `continuous` crossing-diagonal preprocessing, the solved model contains the corresponding merged continuous bars and no frame elements.
- **A pin joint does not transmit member bending moments.** In the in-house solver, truss elements use only the three translational DOFs at each node and carry axial force only. They have no bending, shear, or torsional stiffness.
- Therefore this is not merely “the same frame model with softer joints.” It is an **ideal pin-jointed space-truss model**. Any load path that previously depended on frame bending or rigid-joint action is removed.
- **Rotations at truss-only nodes are numerically restrained by the solver because they are unused DOFs.** This does not make the structural joints rigid: truss members do not use those rotational DOFs, so restraining them adds no member bending stiffness.
- In the **PyNite path**, every member is sent through the existing truss-like representation: local bending rotations `Ry` and `Rz` are released at both ends. Under the present nodal force load cases, this is intended to reproduce the axial-only lattice response as closely as the adapter permits. It remains a numerical cross-check rather than the canonical definition of the truss model.
- `CROSSING_DIAGONAL_MODE = "continuous"` is retained. A geometric crossing of two diagonals is **not** made into a joint unless it is an actual modeled node; split crossing segments are merged back into continuous bars.
- Base/support nodes retain the same support convention as before. “All joints pinned” refers to **member-to-member connection behavior**, not to releasing the tower from its supports.
- No connection eccentricity, gusset/bolt flexibility, slip, initial imperfection, second-order effect, or geometric nonlinearity is included.

A useful consequence of this experiment is diagnostic: if the all-truss model is stable and its axial-force pattern differs substantially from the mixed model, that difference is directly attributable to removing frame/rigid-joint action. If it becomes unstable, that is also informative—it would mean the previous frame stiffness was suppressing a kinematic mechanism in the pin-jointed idealization.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root() -> Path:
    """Find the repository root whether the notebook starts in /notebooks or the repo root."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "solver.py").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the keratia project root containing solver.py and data/.")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results" / "z5_all_pinned_comparison"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from solver import ModelData, read_model, SolveOptions, solve_model, summarize_result
from helpers import plot_signed_stress_3d, plot_deformed_shape
from pynite_adapter import PyNiteRunOptions, analyze_pynite_model, summarize_pynite_result

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RESULTS_DIR:", RESULTS_DIR)

## 1. Inputs and analysis settings

The load cases are unchanged from the previous Z5 comparison notebook. Each load row is applied to one of the two Z5 high load points. Units are N and N·mm.

The only deliberate structural change is the global member idealization: every member becomes axial-only.

In [ ]:
SECTION_MODE = "isotropic"
PRINCIPAL_ORIENTATION = "strong_inward"
CROSSING_DIAGONAL_MODE = "continuous"

DEFORMED_SHAPE_SCALE = 100.0

Z5_CASES = {
    "Case_1": [
        {"node_label": "high_xneg", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 16250.0, "Fy": 8320.0, "Fz": -20000.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_2": [
        {"node_label": "high_xneg", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 48900.0, "Fy": 0.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
    "Case_3": [
        {"node_label": "high_xneg", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
        {"node_label": "high_xpos", "Fx": 21600.0, "Fy": 38000.0, "Fz": -13400.0, "Mx": 0.0, "My": 0.0, "Mz": 0.0},
    ],
}

load_rows = []
for case_name, rows in Z5_CASES.items():
    for row in rows:
        load_rows.append({"case": case_name, **row})

display(pd.DataFrame(load_rows))

In [ ]:
z5_source_model = read_model(
    "Z5",
    DATA_DIR / "Z5_nodes_aligned.csv",
    DATA_DIR / "Z5_elements_with_sections.csv",
)

# Notebook-local all-pinned idealization.
# Preserve the source classification for diagnostics, but solve every member
# as a true truss/bar in the in-house solver.
pinned_elements = z5_source_model.elements.copy()
pinned_elements["original_element_type"] = pinned_elements["element_type"]
pinned_elements["element_type"] = "truss"

z5_model = ModelData(
    name="Z5_all_pinned",
    nodes=z5_source_model.nodes.copy(),
    elements=pinned_elements,
)

print("Input Z5 nodes:", len(z5_source_model.nodes))
print("Input Z5 elements:", len(z5_source_model.elements))
print()
print("Original member types:")
display(z5_source_model.elements["element_type"].value_counts().rename("count").to_frame())
print("Analysis member types after all-pinned conversion:")
display(z5_model.elements["element_type"].value_counts().rename("count").to_frame())

conversion_check = (
    z5_model.elements
    .groupby(["original_element_type", "element_type"])
    .size()
    .rename("count")
    .reset_index()
)
display(conversion_check)

## 2. Shared comparison helpers

The dedicated Case-1-governing table uses the following derived demands:

- `abs_axial_kN`: absolute axial-force magnitude;
- `compression_kN`: compression magnitude only, zero in tension;
- `tension_kN`: tension magnitude only, zero in compression;
- `sigma_abs_MPa`: `sigma_max_abs_MPa`;
- `bending_kNmm`: maximum bending resultant.

An element is reported only when **Case 1 is strictly greater than both Case 2 and Case 3** for at least one of those measures.

In [ ]:
def stacked_member_results(case_results, backend_name):
    """Stack the complete member-force/result table for every case."""
    frames = []
    for case_name, result in case_results.items():
        table = result["member_forces"].copy()
        table.insert(0, "case", case_name)
        table.insert(0, "backend", backend_name)
        frames.append(table)
    return pd.concat(frames, ignore_index=True).sort_values(["case", "element_id"]).reset_index(drop=True)


def _strictly_greater_than_both(c1, c2, c3, rtol=1e-7, atol=1e-9):
    """Avoid classifying floating-point equality as a real exceedance."""
    other_max = max(float(c2), float(c3))
    c1 = float(c1)
    return (c1 > other_max) and (not np.isclose(c1, other_max, rtol=rtol, atol=atol))


def case1_governing_elements(case_results):
    """
    Return only elements where Case 1 exceeds BOTH Case 2 and Case 3
    in at least one physically meaningful demand measure.
    """
    required = {"Case_1", "Case_2", "Case_3"}
    missing = required - set(case_results)
    if missing:
        raise KeyError(f"Missing cases: {sorted(missing)}")

    indexed = {
        case: result["member_forces"].set_index("element_id")
        for case, result in case_results.items()
    }

    common_ids = sorted(
        set(indexed["Case_1"].index)
        & set(indexed["Case_2"].index)
        & set(indexed["Case_3"].index)
    )

    rows = []
    for eid in common_ids:
        r1 = indexed["Case_1"].loc[eid]
        r2 = indexed["Case_2"].loc[eid]
        r3 = indexed["Case_3"].loc[eid]

        def demand_values(r):
            N = float(r["axial_force_signed_N"])
            return {
                "abs_axial_kN": float(r["max_abs_N_N"]) / 1000.0,
                "compression_kN": max(-N, 0.0) / 1000.0,
                "tension_kN": max(N, 0.0) / 1000.0,
                "sigma_abs_MPa": float(r["sigma_max_abs_MPa"]),
                "bending_kNmm": float(r["max_M_resultant_Nmm"]) / 1000.0,
            }

        d1, d2, d3 = demand_values(r1), demand_values(r2), demand_values(r3)

        governing_metrics = [
            metric
            for metric in d1
            if _strictly_greater_than_both(d1[metric], d2[metric], d3[metric])
        ]
        if not governing_metrics:
            continue

        pct_margins = []
        for metric in governing_metrics:
            next_worst = max(d2[metric], d3[metric])
            if abs(next_worst) > 1e-12:
                pct_margins.append((d1[metric] - next_worst) / abs(next_worst) * 100.0)
            elif d1[metric] > 0:
                pct_margins.append(np.inf)

        rows.append({
            "element_id": int(eid),
            "element_type": r1.get("element_type", ""),
            "start_label": r1.get("start_label", ""),
            "end_label": r1.get("end_label", ""),
            "member_role": r1.get("member_role", ""),
            "section_name": r1.get("section_name", ""),
            "governing_metrics": ", ".join(governing_metrics),
            "largest_C1_margin_pct": max(pct_margins) if pct_margins else np.nan,

            "C1_axial_signed_kN": float(r1["axial_force_signed_N"]) / 1000.0,
            "C2_axial_signed_kN": float(r2["axial_force_signed_N"]) / 1000.0,
            "C3_axial_signed_kN": float(r3["axial_force_signed_N"]) / 1000.0,

            "C1_abs_axial_kN": d1["abs_axial_kN"],
            "C2_abs_axial_kN": d2["abs_axial_kN"],
            "C3_abs_axial_kN": d3["abs_axial_kN"],

            "C1_compression_kN": d1["compression_kN"],
            "C2_compression_kN": d2["compression_kN"],
            "C3_compression_kN": d3["compression_kN"],

            "C1_tension_kN": d1["tension_kN"],
            "C2_tension_kN": d2["tension_kN"],
            "C3_tension_kN": d3["tension_kN"],

            "C1_sigma_abs_MPa": d1["sigma_abs_MPa"],
            "C2_sigma_abs_MPa": d2["sigma_abs_MPa"],
            "C3_sigma_abs_MPa": d3["sigma_abs_MPa"],

            "C1_bending_kNmm": d1["bending_kNmm"],
            "C2_bending_kNmm": d2["bending_kNmm"],
            "C3_bending_kNmm": d3["bending_kNmm"],
        })

    if not rows:
        return pd.DataFrame()

    return (
        pd.DataFrame(rows)
        .sort_values(["largest_C1_margin_pct", "element_id"], ascending=[False, True])
        .reset_index(drop=True)
    )

## 3. In-house all-truss solver

This is the canonical implementation of the all-pinned assumption in this notebook. Every member is a true axial-only 3D truss element in `solver.py`.

In [ ]:
inhouse_options = SolveOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
)

z5_inhouse_results = {}
for case_name, loads in Z5_CASES.items():
    print(f"Solving Z5 {case_name} with in-house solver...")
    z5_inhouse_results[case_name] = solve_model(z5_model, loads, inhouse_options)

inhouse_summary = pd.DataFrame([
    {"case": case_name, **summarize_result(result)}
    for case_name, result in z5_inhouse_results.items()
])

display(inhouse_summary)

### 3.1 Case 1 worse than both Case 2 and Case 3 — in-house

Only elements that satisfy the strict three-case governing rule appear below.

In [ ]:
inhouse_case1_governing = case1_governing_elements(z5_inhouse_results)

print("In-house elements where Case 1 governs over BOTH Case 2 and Case 3:")
print("Count:", len(inhouse_case1_governing))
display(inhouse_case1_governing)

### 3.2 All in-house element loads/results

This is the complete `member_forces` output for every element in every case. No elements are filtered out.

In [ ]:
inhouse_all_element_results = stacked_member_results(z5_inhouse_results, "in_house")
display(inhouse_all_element_results)

## 4. PyNite all-pinned cross-check

The exact same all-truss analysis copy and three load cases are passed through `pynite_adapter.py`.

Because every row is now tagged `truss`, the adapter applies its pin-ended truss-like formulation to **every member**. This provides an independent numerical cross-check of the all-pinned lattice, but the in-house solver remains the cleaner mathematical truss formulation.

In [ ]:
pynite_options = PyNiteRunOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
    principal_orientation=PRINCIPAL_ORIENTATION,
)

z5_pynite_results = {}
for case_name, loads in Z5_CASES.items():
    print(f"Solving Z5 {case_name} with PyNite...")
    z5_pynite_results[case_name] = analyze_pynite_model(
        z5_model,
        loads,
        options=pynite_options,
        case_name=case_name,
        combo_name=case_name,
    )

pynite_summary = pd.DataFrame([
    {"case": case_name, **summarize_pynite_result(result)}
    for case_name, result in z5_pynite_results.items()
])

display(pynite_summary)

### 4.1 Case 1 worse than both Case 2 and Case 3 — PyNite

The same governing criteria used for the in-house solver are applied here.

In [ ]:
pynite_case1_governing = case1_governing_elements(z5_pynite_results)

print("PyNite elements where Case 1 governs over BOTH Case 2 and Case 3:")
print("Count:", len(pynite_case1_governing))
display(pynite_case1_governing)

### 4.2 All PyNite element loads/results

This is the complete member-results table returned by the current PyNite adapter for all elements and all three cases.

In [ ]:
pynite_all_element_results = stacked_member_results(z5_pynite_results, "PyNite")
display(pynite_all_element_results)

## 5. Direct comparison of the Case-1-governing element sets

This section makes it easy to see whether the two solver paths identify the same elements as being more highly loaded in Case 1 than in **both** alternatives.

In [ ]:
inhouse_ids = set(inhouse_case1_governing["element_id"]) if not inhouse_case1_governing.empty else set()
pynite_ids = set(pynite_case1_governing["element_id"]) if not pynite_case1_governing.empty else set()
all_governing_ids = sorted(inhouse_ids | pynite_ids)

comparison_rows = []
for eid in all_governing_ids:
    ih = inhouse_case1_governing.loc[inhouse_case1_governing["element_id"] == eid]
    py = pynite_case1_governing.loc[pynite_case1_governing["element_id"] == eid]

    source = ih.iloc[0] if not ih.empty else py.iloc[0]
    comparison_rows.append({
        "element_id": eid,
        "element_type": source.get("element_type", ""),
        "start_label": source.get("start_label", ""),
        "end_label": source.get("end_label", ""),
        "member_role": source.get("member_role", ""),
        "section_name": source.get("section_name", ""),
        "inhouse_C1_governs": eid in inhouse_ids,
        "pynite_C1_governs": eid in pynite_ids,
        "inhouse_governing_metrics": "" if ih.empty else ih.iloc[0]["governing_metrics"],
        "pynite_governing_metrics": "" if py.empty else py.iloc[0]["governing_metrics"],
    })

governing_set_comparison = pd.DataFrame(comparison_rows)
display(governing_set_comparison)

print("Common governing elements:", sorted(inhouse_ids & pynite_ids))
print("In-house only:", sorted(inhouse_ids - pynite_ids))
print("PyNite only:", sorted(pynite_ids - inhouse_ids))

## 6. Save comparison tables

The notebook writes the principal tables to `results/z5_all_pinned_comparison/` so they can be inspected without re-running the display cells.

In [ ]:
inhouse_summary.to_csv(RESULTS_DIR / "inhouse_summary.csv", index=False)
pynite_summary.to_csv(RESULTS_DIR / "pynite_summary.csv", index=False)

inhouse_all_element_results.to_csv(
    RESULTS_DIR / "inhouse_all_element_results.csv", index=False
)
pynite_all_element_results.to_csv(
    RESULTS_DIR / "pynite_all_element_results.csv", index=False
)

inhouse_case1_governing.to_csv(
    RESULTS_DIR / "inhouse_case1_governs_both.csv", index=False
)
pynite_case1_governing.to_csv(
    RESULTS_DIR / "pynite_case1_governs_both.csv", index=False
)
governing_set_comparison.to_csv(
    RESULTS_DIR / "case1_governing_set_backend_comparison.csv", index=False
)

print("Wrote comparison CSVs to:", RESULTS_DIR)

## 7. Whole-construction tension/compression visualizations

The plots below use **axial signed stress** so that the color directly represents the axial load path:

- positive = tension;
- negative = compression.

Each figure is autoscaled symmetrically around zero by the existing plotting helper. Therefore use the numerical tables above—not color intensity alone—for quantitative comparison between different cases.

In [ ]:
for case_name, result in z5_inhouse_results.items():
    fig, ax = plot_signed_stress_3d(
        result["nodes"],
        result["elements"],
        result["member_forces"],
        title=f"Z5 all-pinned in-house axial tension/compression — {case_name}",
        stress_column="sigma_axial_signed_MPa",
        annotate_load_points=True,
        annotate_elements=False,
        show_frame=True,
        show_truss=True,
    )
    plt.show()

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_signed_stress_3d(
        result["nodes"],
        result["elements"],
        result["member_forces"],
        title=f"Z5 all-pinned PyNite axial tension/compression — {case_name}",
        stress_column="sigma_axial_signed_MPa",
        annotate_load_points=True,
        annotate_elements=False,
        show_frame=True,
        show_truss=True,
    )
    plt.show()

### Optional: signed controlling normal stress

In this all-truss experiment, the in-house members have no bending stiffness, so `sigma_extreme_signed_MPa` should collapse to the axial stress. The PyNite truss-like path should show the same behavior apart from numerical noise.

This plot is retained mainly for consistency with the previous comparison notebook; the axial signed-stress plots above are the more direct view of the load path.

In [ ]:
for backend_name, case_results in [
    ("in-house", z5_inhouse_results),
    ("PyNite", z5_pynite_results),
]:
    for case_name, result in case_results.items():
        fig, ax = plot_signed_stress_3d(
            result["nodes"],
            result["elements"],
            result["member_forces"],
            title=f"Z5 {backend_name} controlling signed normal stress — {case_name}",
            stress_column="sigma_extreme_signed_MPa",
            annotate_load_points=True,
            annotate_elements=False,
            show_frame=True,
            show_truss=True,
        )
        plt.show()

## 8. Whole-construction deformation visualizations

Undeformed geometry is shown faintly and the deformed shape is exaggerated by `DEFORMED_SHAPE_SCALE` for visibility.

In [ ]:
for case_name, result in z5_inhouse_results.items():
    fig, ax = plot_deformed_shape(
        result["nodes"],
        result["elements"],
        result["displacements"],
        scale=DEFORMED_SHAPE_SCALE,
        title=f"Z5 all-pinned in-house deformed shape — {case_name}",
        show_truss=True,
    )
    plt.show()

In [ ]:
for case_name, result in z5_pynite_results.items():
    fig, ax = plot_deformed_shape(
        result["nodes"],
        result["elements"],
        result["displacements"],
        scale=DEFORMED_SHAPE_SCALE,
        title=f"Z5 all-pinned PyNite deformed shape — {case_name}",
        show_truss=True,
    )
    plt.show()

## 9. Interpretation notes

For this all-pinned model:

- `compression_kN` is the direct member compression demand and is the relevant force quantity for later buckling screening;
- `tension_kN` is the direct tensile axial demand;
- `sigma_abs_MPa` should be governed by axial stress because the in-house model contains no frame bending;
- `bending_kNmm` should be zero for the true truss solution and is retained only as a diagnostic/check of the PyNite translation.

Do **not** interpret a raw signed value such as `-10 MPa > -80 MPa` as Case 1 being worse. In compression, `-80 MPa` is the larger compressive demand.

The important comparison with notebook 05 is conceptual: notebook 05 allows rigid frame joints for members classified as frames, while this notebook removes that mechanism completely and asks whether a purely pin-jointed lattice can carry the same nodal loads.

This remains an exploratory first-order linear-elastic analysis. It does not by itself constitute a code buckling, connection, or capacity check.